# Day 080 — Exercise 3: Acting and Observing

**What you'll build:** `execute_action` (runs the tool a step names) and `call_llm` (the injectable model call, from Day 79).

**Why it matters:** the *observe* half of ReAct. `execute_action` produces the result that becomes the next `Observation` — the fact the model reasons over on its next turn.

In [ ]:
import json

def _make_mock_llm(script):
    """Return an llm_fn(messages) that yields each scripted reply in turn.

    Repeats the last reply once the script is exhausted - handy for testing a
    runaway loop (a model that never emits a Final Answer).
    """
    state = {'i': 0}
    def _fn(messages):
        i = state['i']
        state['i'] = min(i + 1, len(script) - 1)
        return script[i]
    return _fn
import ast
import json
import operator

# ── tools reused from Day 79: a safe calculator + a fact-lookup tool ──────────
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def _eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError("unsupported expression")


def safe_calculate(expression):
    """Evaluate arithmetic without eval() (see Day 79)."""
    return _eval_node(ast.parse(expression, mode="eval").body)


_FACTS = {
    "speed of light": "299792458 m/s",
    "pi": "3.14159",
    "earth radius": "6371 km",
    "days in a year": "365",
}


def _lookup(args):
    query = str(args.get("query", "")).lower().strip()
    for key, value in _FACTS.items():
        if query and (query in key or key in query):
            return value
    return "No result found for " + repr(args.get("query", ""))


DEFAULT_TOOLS = {
    "calculator": {
        "description": "Evaluate an arithmetic expression, e.g. 2 * (3 + 4).",
        "parameters": {"expression": "string - the arithmetic to evaluate"},
        "fn": lambda args: str(safe_calculate(args["expression"])),
    },
    "lookup": {
        "description": "Look up a known fact: speed of light, pi, earth radius, "
                       "days in a year.",
        "parameters": {"query": "string - what to look up"},
        "fn": _lookup,
    },
}


def build_tool_descriptions(tools):
    """Render a tool registry as prompt text (Day 79)."""
    lines = []
    for name, spec in tools.items():
        params = ", ".join(spec.get("parameters", {}))
        lines.append("- " + name + "(" + params + "): " + spec["description"])
    return "\n".join(lines)


def safe_parse_json(text):
    """Slice first '{' to last '}' and parse. Returns dict|None (Day 79)."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None

# ── parsing the ReAct format ──────────────────────────────────────────────────
def _line_value(text, prefix):
    """Text after the first line starting with prefix (case-insensitive), else ''."""
    for line in text.splitlines():
        if line.strip().lower().startswith(prefix.lower()):
            return line.strip()[len(prefix):].strip()
    return ""


def _after_marker(text, marker):
    """Everything after marker (case-insensitive), or None if absent."""
    idx = text.lower().find(marker.lower())
    if idx == -1:
        return None
    return text[idx + len(marker):].strip()


def parse_react_step(text):
    """Parse one ReAct step. NEVER raises.

    Returns either:
      {"type": "action", "thought": str, "tool": str, "input": dict}
      {"type": "final",  "thought": str, "answer": str}
    A reply with no recognisable Action falls back to a final answer holding
    the raw text - so a malformed step still ends the loop cleanly.
    """
    thought = _line_value(text, "Thought:")
    final = _after_marker(text, "Final Answer:")
    if final is not None:
        return {"type": "final", "thought": thought, "answer": final}
    action = _line_value(text, "Action:")
    if action:
        args = safe_parse_json(_line_value(text, "Input:")) or {}
        return {"type": "action", "thought": thought, "tool": action, "input": args}
    return {"type": "final", "thought": thought, "answer": text.strip()}

# ── formatting the trace (the scratchpad) ─────────────────────────────────────
def format_step(step):
    """Render an action step back into ReAct text for the scratchpad."""
    return ("Thought: " + step["thought"] + "\n"
            + "Action: " + step["tool"] + "\n"
            + "Input: " + json.dumps(step["input"]))


def format_observation(result):
    """Render a tool result as an Observation line."""
    return "Observation: " + str(result)


def build_react_prompt(task, tools, scratchpad):
    """Build the [system, user] messages for one ReAct step."""
    system = "\n".join([
        "You are a reasoning agent. Solve the task step by step using the "
        "ReAct format: reason, act, observe, repeat.",
        "",
        "Available tools:",
        build_tool_descriptions(tools),
        "",
        "On each turn reply in EXACTLY this format:",
        "Thought: <your reasoning about what to do next>",
        "Action: <one tool name from the list above>",
        'Input: {"<param>": "<value>"}',
        "",
        "You will then receive an Observation with the tool's result.",
        "When you can answer, reply instead with:",
        "Thought: <your final reasoning>",
        "Final Answer: <the answer>",
    ])
    user = "Task: " + str(task)
    if scratchpad:
        user = user + "\n\n" + scratchpad.rstrip()
    user = user + "\n\nThought:"
    return [{"role": "system", "content": system},
            {"role": "user", "content": user}]


## Task

1. `execute_action(step, tools) -> str` — look up `step['tool']`; unknown → an `Error: unknown tool ...` string; else call `tools[name]['fn'](step.get('input', {}))` in `try/except`, returning the result (or error text) as a string. **Never raises.**
2. `call_llm(messages, llm_fn=None) -> str` — `llm_fn(messages)` if given, else `ollama.chat(model='llama3.2', messages=messages)['message']['content']`.

## Your Implementation

In [ ]:
def execute_action(step, tools):
    """Run the tool named in a ReAct action step. Returns a string. Never raises."""
    raise NotImplementedError

def call_llm(messages, llm_fn=None):
    """Call the chat model, or the injected llm_fn(messages) -> str."""
    raise NotImplementedError


In [ ]:

# ── acting + calling the model ────────────────────────────────────────────────
def execute_action(step, tools):
    """Run the tool named in a ReAct action step. Returns a string, never raises."""
    name = step.get("tool")
    if name not in tools:
        return "Error: unknown tool " + repr(name) + ". Available: " + ", ".join(tools)
    try:
        return str(tools[name]["fn"](step.get("input", {})))
    except Exception as exc:
        return "Error running " + str(name) + ": " + str(exc)


def call_llm(messages, llm_fn=None):
    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]


## Automated checks

In [ ]:

score, total = 0, 5
try:
    r = execute_action({'tool': 'calculator', 'input': {'expression': '6*7'}}, DEFAULT_TOOLS)
    assert r == '42'
    score += 1; print("✅ execute_action runs a tool")

    r2 = execute_action({'tool': 'nope', 'input': {}}, DEFAULT_TOOLS)
    assert 'unknown tool' in r2.lower()
    score += 1; print("✅ execute_action reports unknown tools (no crash)")

    r3 = execute_action({'tool': 'calculator', 'input': {}}, DEFAULT_TOOLS)
    assert 'error' in r3.lower()
    score += 1; print("✅ execute_action captures tool errors (never raises)")

    r4 = execute_action({'tool': 'lookup', 'input': {'query': 'pi'}}, DEFAULT_TOOLS)
    assert '3.14' in r4
    score += 1; print("✅ the lookup tool returns a known fact")

    got = call_llm([{'role': 'user', 'content': 'hi'}], llm_fn=lambda m: 'X')
    assert got == 'X'
    score += 1; print("✅ call_llm uses the injected llm_fn")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── acting + calling the model ────────────────────────────────────────────────
def execute_action(step, tools):
    """Run the tool named in a ReAct action step. Returns a string, never raises."""
    name = step.get("tool")
    if name not in tools:
        return "Error: unknown tool " + repr(name) + ". Available: " + ", ".join(tools)
    try:
        return str(tools[name]["fn"](step.get("input", {})))
    except Exception as exc:
        return "Error running " + str(name) + ": " + str(exc)


def call_llm(messages, llm_fn=None):
    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]
```

**Why does a tool error become the Observation?** In ReAct the model reads each Observation and reasons about it. An error it can *see* (`Error running calculator: ...`) is something it can reason its way around; an exception just ends the run.

</details>